# Bounded indexed multi-rate loading benchmark

Measured on the expanded $100B corpus, restoring the same epoch-one batch-50 checkpoint, optimizer and RNG state. Three identical 32-sample GPU batches per mode. Baseline averaged 72.99 seconds/batch; indexed loading including first-use builds averaged 30.19 seconds (2.42x faster); reuse averaged 3.52 seconds (20.76x faster). Every measured loss matched exactly. Twelve real-data window comparisons across all four rates matched values, masks and dates exactly. Peak process RSS was 10.18 GiB baseline, 14.85 GiB initial indexing and 12.44 GiB reuse. These are short controlled measurements, not whole-epoch timing.

Production resumed from the pinned checkpoint using `artifacts/multirate_recovery/100B/resume_indexed_loading_v9.py`. It retains the same model, optimizer, precision, batch size, targets and epoch backtests. Caches contain normalized observations only, keyed by corpus fingerprints, fit normalization and feature layout. Polars builds one issuer at a time (128 MiB tensor limit), Torch memory-maps the indexes; larger issuers fall back to bounded calendar windows.

For a controlled rerun, isolate the GPU from other active training. The benchmark checkpoint and measurements live under `artifacts/multirate_recovery/100B/indexed_loading_benchmark`. The next cell restores the baseline source from its recorded repository revision. Choose `baseline`, `indexed_disk_cold` or `indexed_disk_warm`; cold requires a fresh compatible index directory. Existing caches make a rerun warm regardless of its label.


In [ ]:
from pathlib import Path
import subprocess
r = Path('artifacts/multirate_recovery/100B/indexed_loading_benchmark')
baseline = subprocess.run(['git','show','528535a:quant_orchestrator/research_tools/streaming_context.py'], check=True, capture_output=True, text=True).stdout
(r/'streaming_context_before.py').write_text(baseline)
mode = 'indexed_disk_warm'


In [ ]:
import sys,json,runpy,importlib.util,time,resource
from pathlib import Path
import torch
root=Path('artifacts/multirate_recovery/100B').resolve();out=root/'indexed_loading_benchmark'/mode;out.mkdir(exist_ok=True)
if mode=='baseline':
 import quant_orchestrator.research_tools.streaming_context as context
 spec=importlib.util.spec_from_file_location('old_context',str(root/'indexed_loading_benchmark/streaming_context_before.py'));old=importlib.util.module_from_spec(spec);spec.loader.exec_module(old)
 for cls in [old.StreamingContext,old.StreamingFamilyContext]:
  original_init=cls.__init__
  def init(self,*args,_original=original_init,**kwargs):
   kwargs.pop('index_directory',None);_original(self,*args,**kwargs)
  cls.__init__=init
 context.StreamingContext=old.StreamingContext;context.StreamingFamilyContext=old.StreamingFamilyContext
from quant_orchestrator.platforms.ml_frameworks.torch.models.transformers.multirate import Trainer
original=Trainer.fit
records=[]
def fit(self,epochs,step,**kwargs):
 lazy=step.__globals__['_LazySample'];materialize=lazy._materialize
 material_seconds=0.;backward_seconds=0.
 def timed_material(sample):
  nonlocal material_seconds
  t=time.perf_counter();result=materialize(sample);material_seconds+=time.perf_counter()-t;return result
 lazy._materialize=timed_material
 old_back=self.backward_step
 def back(*a,**kw):
  nonlocal backward_seconds
  torch.cuda.synchronize();t=time.perf_counter();result=old_back(*a,**kw);torch.cuda.synchronize();backward_seconds+=time.perf_counter()-t;return result
 self.backward_step=back
 torch.cuda.synchronize();last=time.perf_counter()
 def callback(epoch,batch,total,loss):
  nonlocal last,material_seconds,backward_seconds
  torch.cuda.synchronize();now=time.perf_counter()
  r=dict(epoch=epoch,batch=batch,loss=loss,seconds=now-last,materialize_seconds=material_seconds,backward_seconds=backward_seconds)
  records.append(r);print(json.dumps(r),flush=True);last=now;material_seconds=backward_seconds=0.
  (out/'measurements.json').write_text(json.dumps({'mode':mode,'batches':records,'peak_rss_mib':resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024},indent=2))
  if len(records)>=3:raise SystemExit(0)
 kwargs['on_batch_end']=callback
 return original(self,epochs,step,**kwargs)
Trainer.fit=fit
cmd=json.loads((root/'training_command_expanded_v9.json').read_text())
cmd[cmd.index('--output-dir')+1]=str(out)
i=cmd.index('--epoch-evaluation-dir');del cmd[i:i+2]
cmd+=['--checkpoint',str(root/'indexed_loading_benchmark/starting_checkpoint.pt'),'--resume-training','--skip-predictions']
sys.argv=cmd[1:]
try:
 runpy.run_path(cmd[1],run_name='__main__')
except SystemExit as exc:
 if exc.code not in (0, None):raise
